The auto encoder is a commonly used unsupervised algorithm to perform dimension reduction or denoising. It is in general applied on images. In this tutorial, we will build a simple autoencoder and a variational one to showcase the differences between them and what they can achieve.

In [ ]:
# install torch and torchvision
!uv sync --extra deep-learning

Defaulting to user installation because normal site-packages is not writeable
ERROR: Directory '.[deep-learning]' is not installable. Neither 'setup.py' nor 'pyproject.toml' found.


I - Autoencoder

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import plotly.express as px

# load MNIST data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # flatten 28x28= 784
])

mnist = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
loader = DataLoader(mnist, batch_size=256, shuffle=True)

# define our autoencoder class
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3)       # 3D bottleneck
        )
        self.decoder = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z

autoencoder = Autoencoder()
optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Main training loop
epochs = 20
autoencoder.train()

for epoch in range(epochs):
    total_loss = 0
    for batch, _ in loader:
        optimizer.zero_grad()
        recon, _ = autoencoder(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.4f}")

# encode the dataset
autoencoder.eval()
all_data = mnist.data.float().view(-1, 784) / 255.0
all_labels = mnist.targets.numpy()

with torch.no_grad():
    z = autoencoder.encoder(all_data).numpy()

# plot
fig = px.scatter_3d(
    x=z[:, 0], y=z[:, 1], z=z[:, 2],
    color=all_labels.astype(str),
    title="MNIST Autoencoder 3D Embedding",
    opacity=0.7
)
fig.update_traces(marker=dict(size=3))
fig.show()


II - Variational Autoencoder

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import plotly.express as px


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))   # flatten
])

mnist = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
loader = DataLoader(mnist, batch_size=256, shuffle=True)

# VAE
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(64, 3)     # 3D latent mean
        self.fc_logvar = nn.Linear(64, 3) # 3D latent log variance

        self.decoder = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

vae = VAE()

optimizer = optim.Adam(vae.parameters(), lr=1e-3)

# the loss is MSE + a KL divergence to force a latent space
def vae_loss(recon, x, mu, logvar):
    mse = nn.functional.mse_loss(recon, x, reduction='sum')
    # KL divergence forcing latent space ~ N(0,1)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (mse + kld) / x.size(0)

# train
epochs = 20
vae.train()

for epoch in range(epochs):
    total = 0
    for batch, _ in loader:
        optimizer.zero_grad()
        recon, mu, logvar, _ = vae(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"Epoch {epoch+1}/{epochs} – Loss: {total/len(loader):.4f}")

# let's encode again with out VAE this time around
vae.eval()
all_data = mnist.data.float().view(-1, 784) / 255.0
all_labels = mnist.targets.numpy()

with torch.no_grad():
    mu, logvar = vae.encode(all_data)
    z = mu.numpy()  # use mu for visualization

# and plot
fig = px.scatter_3d(
    x=z[:, 0], y=z[:, 1], z=z[:, 2],
    color=all_labels.astype(str),
    title="MNIST 3D Latent Space VAE",
    opacity=0.7
)
fig.update_traces(marker=dict(size=3))
fig.show()


III - Your turn !

- Can you think of other dimension reduction algorithms? 
Implement one to compare the results.
- You can interpolate in the latent space. Implement a function to do so and compare the results with a simple autoencoder and with a VAE. You can for instance implement a morphing of images that way.
- What is your conclusion on the usefulness of embeddings ?
- You can try to implement it with CIFAR-10 or any other vision dataset.